# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Multi-Period Portfolio Optimization**

\begin{align}
    \min \quad & \sum_{t \in T} \left( \sum_{i=1}^n \sum_{j=1}^n Q_{t,ij} x_{t,i} x_{t,j} + \lambda \sum_{i=1}^n \delta_{t,i}^2 \right) \\
    \text{subject to} \quad
    & \sum_{i=1}^n x_{t,i} = 1 \quad && \forall t \in T \quad \text{(Budget)} \\
    & \sum_{i=1}^n y_{t,i} \leq K \quad && \forall t \in T \quad \text{(Cardinality)} \\
    & x_{t,i} \leq y_{t,i} \quad && \forall t \in T,\ \forall i \in I \quad \text{(Linking)} \\
    & x_{t,i} \geq 0.01 \cdot y_{t,i} \quad && \forall t \in T,\ \forall i \in I \quad \text{(Min investment)} \\
    & \delta_{t,i} = x_{t,i} - x_{t-1,i} \quad && \forall t > 1,\ \forall i \in I \quad \text{(Trade update)} \\
    & x_{t,i} \geq 0,\ y_{t,i} \in \{0,1\},\ \delta_{t,i} \in \mathbb{R} \quad && \forall t,\ i \quad \text{(Domain)}
\end{align}

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
MPCCPOP12\_10 & MBQP & Convex & 2160 & 720 & 0 & 1585 \\
MPCCPOP12\_20 & MBQP & Convex & 4320 & 1440 & 0 & 3025 \\
MPCCPOP12\_30 & MBQP & Convex & 6480 & 2160 & 0 & 4465\\
MPCCPOP12\_50 & MBQP & Convex & 10800 & 3600 & 0 & 7345\\
\end{array}
$$


#  **Solver Setup**


## MPCCPOP12\_10

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO",
]# 10


tickers = list(dict.fromkeys(tickers))  # Removes duplicates while preserving order
data = yf.download(tickers, start='2019-06-21', end='2025-06-26')['Close']
returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = 6 * 12
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]

# Initialize GAMSPy Container and Sets
m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

t = Set(m, name="t", records=time_periods)
tt = Alias(m, name="tt", alias_with=t)

# Parameters
Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
q_records = []
for t_idx, tp in enumerate(time_periods):
    cov = cov_list[t_idx]
    q_records += [(tp, assets[r], assets[c], cov[r, c])
                  for r in range(len(assets)) for c in range(len(assets))]

# Convert to DataFrame with categorical columns
df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
Q_t.records = df_q

# Variables
x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
budget = Equation(m, name="budget", domain=[t])
budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
cardinality = Equation(m, name="cardinality", domain=[t])
cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
if len(time_periods) > 1:
    link_trade = Equation(m, name="link_trade", domain=[t, i])
    for t_idx in range(1, len(time_periods)):
        curr_t = time_periods[t_idx]
        prev_t = time_periods[t_idx - 1]
        for asset in assets:
            link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

# Link x and y (e.g. enforce x[t, i] <= y[t, i])
link_x_y = Equation(m, name="link_x_y", domain=[t, i])
link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
invest = Equation(m, name="invest", domain=[t, i])
invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Objective (minimize risk + transaction costs)
transaction_cost = Sum([t, i], delta_x[t, i] * delta_x[t, i]) * 0.0002
portfolio_variance = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])


obj = portfolio_variance + transaction_cost

# Model and Solve
model = Model(m, name="MPCCPOP12_10", equations=m.getEquations(),
              problem="MIQCP", sense=Sense.MIN, objective=obj)

import sys
model.solve(output=sys.stdout, solver = "CONVERT")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## MPCCPOP12\_20

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data

tickers = [
    "ADBE", "CMCSA", "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO"
]
tickers = list(dict.fromkeys(tickers))  # Removes duplicates while preserving order
data = yf.download(tickers, start='2019-06-21', end='2025-06-26')['Close']
returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = 6 * 12
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]

# Initialize GAMSPy Container and Sets
m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

t = Set(m, name="t", records=time_periods)
tt = Alias(m, name="tt", alias_with=t)

# Parameters
Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
q_records = []
for t_idx, tp in enumerate(time_periods):
    cov = cov_list[t_idx]
    q_records += [(tp, assets[r], assets[c], cov[r, c])
                  for r in range(len(assets)) for c in range(len(assets))]

# Convert to DataFrame with categorical columns
df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
Q_t.records = df_q

# Variables
x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
budget = Equation(m, name="budget", domain=[t])
budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
cardinality = Equation(m, name="cardinality", domain=[t])
cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
if len(time_periods) > 1:
    link_trade = Equation(m, name="link_trade", domain=[t, i])
    for t_idx in range(1, len(time_periods)):
        curr_t = time_periods[t_idx]
        prev_t = time_periods[t_idx - 1]
        for asset in assets:
            link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

# Link x and y (e.g. enforce x[t, i] <= y[t, i])
link_x_y = Equation(m, name="link_x_y", domain=[t, i])
link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
invest = Equation(m, name="invest", domain=[t, i])
invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Objective (minimize risk + transaction costs)
transaction_cost = Sum([t, i], delta_x[t, i] * delta_x[t, i]) * 0.0002
portfolio_variance = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])

obj = portfolio_variance + transaction_cost

# Model and Solve
model = Model(m, name="MPCCPOP12_20", equations=m.getEquations(),
              problem="MIQCP", sense=Sense.MIN, objective=obj)

import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})
#, "Model.Convexity.AssumeConvex": True

In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

## MPCCPOP12\_30

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data
tickers = [
    "ADBE", "CMCSA", "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE"
]# 30


tickers = list(dict.fromkeys(tickers))  # Removes duplicates while preserving order
data = yf.download(tickers, start='2019-06-21', end='2025-06-26')['Close']
returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = 6 * 12
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]

# Initialize GAMSPy Container and Sets
m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

t = Set(m, name="t", records=time_periods)
tt = Alias(m, name="tt", alias_with=t)

# Parameters
Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
q_records = []
for t_idx, tp in enumerate(time_periods):
    cov = cov_list[t_idx]
    q_records += [(tp, assets[r], assets[c], cov[r, c])
                  for r in range(len(assets)) for c in range(len(assets))]

# Convert to DataFrame with categorical columns
df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
Q_t.records = df_q

# Variables
x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
budget = Equation(m, name="budget", domain=[t])
budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
cardinality = Equation(m, name="cardinality", domain=[t])
cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
if len(time_periods) > 1:
    link_trade = Equation(m, name="link_trade", domain=[t, i])
    for t_idx in range(1, len(time_periods)):
        curr_t = time_periods[t_idx]
        prev_t = time_periods[t_idx - 1]
        for asset in assets:
            link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

# Link x and y (e.g. enforce x[t, i] <= y[t, i])
link_x_y = Equation(m, name="link_x_y", domain=[t, i])
link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
invest = Equation(m, name="invest", domain=[t, i])
invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Objective (minimize risk + transaction costs)
transaction_cost = Sum([t, i], delta_x[t, i] * delta_x[t, i]) * 0.0002
portfolio_variance = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])


obj = portfolio_variance + transaction_cost

# Model and Solve
model = Model(m, name="MPCCPOP12_30", equations=m.getEquations(),
              problem="MIQCP", sense=Sense.MIN, objective=obj)

import sys
model.solve(output=sys.stdout, solver = "CONVERT")

## MPCCPOP12\_50

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download data
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "PYPL", "INTC", "CSCO",
    "ADBE", "CMCSA", "PEP", "NFLX", "TXN", "QCOM", "AVGO", "AMD", "SBUX", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "PG", "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS"
]# 50


tickers = list(dict.fromkeys(tickers))  # Removes duplicates while preserving order
data = yf.download(tickers, start='2019-06-21', end='2025-06-26')['Close']
returns = data.pct_change().dropna()
tickers_cleaned = returns.columns.tolist()
assets = [f"asset_{i+1}" for i in range(len(tickers_cleaned))]

# Compute time-varying covariances
T = 6 * 12
time_periods = [f"t{t+1}" for t in range(T)]
split_returns = np.array_split(returns, T)
cov_list = [returns.cov().values for r in split_returns]

# Initialize GAMSPy Container and Sets
m = Container()

#assets = [f"asset_{i+1}" for i in range(len(tickers))]
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

t = Set(m, name="t", records=time_periods)
tt = Alias(m, name="tt", alias_with=t)

# Parameters
Q_t = Parameter(m, name="Q_t", domain=[t, i, ii])

# Build all Q_t records
q_records = []
for t_idx, tp in enumerate(time_periods):
    cov = cov_list[t_idx]
    q_records += [(tp, assets[r], assets[c], cov[r, c])
                  for r in range(len(assets)) for c in range(len(assets))]

# Convert to DataFrame with categorical columns
df_q = pd.DataFrame(q_records, columns=["t", "i", "ii", "value"])
df_q["t"] = pd.Categorical(df_q["t"], categories=time_periods)
df_q["i"] = pd.Categorical(df_q["i"], categories=assets)
df_q["ii"] = pd.Categorical(df_q["ii"], categories=assets)
Q_t.records = df_q

# Variables
x = Variable(m, name="x", domain=[t, i], type="Positive")  # weights
y = Variable(m, name="y", domain=[t, i], type="Binary")     # selection
delta_x = Variable(m, name="delta_x", domain=[t, i], type="Free")  # trades

# Constraints

# Budget: sum of weights = 1 per period
budget = Equation(m, name="budget", domain=[t])
budget[t] = Sum(i, x[t, i]) == 1

# Cardinality: max K assets per period
cardinality = Equation(m, name="cardinality", domain=[t])
cardinality[t] = Sum(i, y[t, i]) <= 12

# Linking trades over time (only if there's more than one period)
if len(time_periods) > 1:
    link_trade = Equation(m, name="link_trade", domain=[t, i])
    for t_idx in range(1, len(time_periods)):
        curr_t = time_periods[t_idx]
        prev_t = time_periods[t_idx - 1]
        for asset in assets:
            link_trade[curr_t, asset] = delta_x[curr_t, asset] == x[curr_t, asset] - x[prev_t, asset]

# Link x and y (e.g. enforce x[t, i] <= y[t, i])
link_x_y = Equation(m, name="link_x_y", domain=[t, i])
link_x_y[t, i] = x[t, i] <= y[t, i]

# Invest
invest = Equation(m, name="invest", domain=[t, i])
invest[t, i] = x[t, i] >= 0.01 * y[t, i]

# Objective (minimize risk + transaction costs)
transaction_cost = Sum([t, i], delta_x[t, i] * delta_x[t, i]) * 0.0002
portfolio_variance = Sum([t, i, ii], Q_t[t, i, ii] * x[t, i] * x[t, ii])


obj = portfolio_variance + transaction_cost

# Model and Solve
model = Model(m, name="MPCCPOP12_50", equations=m.getEquations(),
              problem="MIQCP", sense=Sense.MIN, objective=obj)

import sys
model.solve(output=sys.stdout, solver = "CONVERT")